In [0]:
%run ./config

In [0]:
%run ./functions

In [0]:
# %python
from pyspark.sql import SparkSession
from datetime import datetime, timezone
import json
import uuid
import time

spark = SparkSession.builder.getOrCreate()

# 🔹 Função auxiliar para logar (capturada pelo Log Analytics)
def log_event(message, severity="INFO"):
    trace_id = str(uuid.uuid4())  # 🔹 Identificador único para correlação
    log = {
        "timestamp": datetime.now(timezone.utc).isoformat(),  # ✅ compatível com Python 3.12+
        "trace_id": trace_id,
        "severity": severity,
        "message": message
    }
    print(json.dumps(log))  # Databricks envia automaticamente para Log Analytics

# 🔹 Teste de conexão e validação dos segredos
try:
    blob_service_client, container_name = get_blob_service_client()
    log_event(f"✅ Conectado ao container: {container_name}")
except Exception as e:
    log_event(f"❌ Erro na conexão com Azure Storage: {e}", severity="ERROR")
    raise

# 🔹 Execução principal
try:
    result = execute_batch(
        blob_service_client=blob_service_client,
        container_name=container_name,
        records=5,
        folder="inep_alunos_simulado"
    )

    log_event(f"✅ {result['records_sent']} registros enviados para {result['blob_name']}")

    # 🔹 Verificação de integridade técnica
    df_preview = result["preview"]
    record_count = len(df_preview)

    if record_count == 0:
        log_event("❌ Nenhum registro carregado na camada Bronze.", severity="ERROR")
        raise ValueError("Nenhum registro carregado na camada Bronze.")

    expected_columns = ["ano","id_municipio","id_escola","id_aluno","caderno","serie","rede","presenca","preenchimento_caderno",
    "alfabetizado","proficiencia","peso_aluno"]                  
    missing_cols = [c for c in expected_columns if c not in df_preview.columns]

    if missing_cols:
        log_event(f"❌ Colunas ausentes: {missing_cols}", severity="ERROR")
        raise ValueError(f"Colunas ausentes: {missing_cols}")

    log_event(f"📊 Concluído. Schema válido e {record_count} registros carregados.")

except Exception as e:
    log_event(f"❌ Erro na execução: {e}", severity="ERROR")

# 🔹 Ciclos de streaming
for ciclo in range(2):
    try:
        result = execute_batch(
            blob_service_client=blob_service_client,
            container_name=container_name,
            records=10,
            folder="inep_alunos_streaming"
        )

        log_event(f"✅ Lote {ciclo + 1} enviado: {result['blob_name']}")
        df_preview = result["preview"]
        record_count = len(df_preview)

        if record_count == 0:
            log_event(f"⚠️ Lote {ciclo + 1} vazio.", severity="WARNING")
        else:
            log_event(f"📦 Lote {ciclo + 1} contém {record_count} registros.")

        expected_columns = ["ano","id_municipio","id_escola","id_aluno","caderno","serie","rede","presenca","preenchimento_caderno",
    "alfabetizado","proficiencia","peso_aluno"]
        missing_cols = [c for c in expected_columns if c not in df_preview.columns]
        if missing_cols:
            log_event(f"⚠️ Colunas ausentes no lote {ciclo + 1}: {missing_cols}", severity="WARNING")

    except Exception as e:
        log_event(f"❌ Erro no lote {ciclo + 1}: {e}", severity="ERROR")

    time.sleep(10)
